# Kapitel 5 – Dimensionsreducering

Individuella övningsuppgifter.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path

## Fråga 1

### Vad menas med curse of dimensionality?

**Curse of dimensionality** innebär att problem kan uppstå när ett dataset får väldigt många features.

Ju fler dimensioner vi har, desto mer utspridd blir datan och desto mer data kan behövas för att hitta tydliga mönster.

Det kan också göra modeller långsammare och öka risken för att modellen lär sig brus istället för användbara samband.

## Fråga 2

### Vad är dimensionsreducering och varför görs det?

Dimensionsreducering innebär att man minskar antalet features i ett dataset men försöker behålla så mycket viktig information som möjligt.

Det kan användas för att:

- göra modeller snabbare
- minska mängden brus
- förenkla visualisering
- minska problem som kan uppstå när datan har väldigt många dimensioner

PCA är en vanlig metod för detta.

## Fråga 3

### Förklara översiktligt hur PCA fungerar. Använd figur 5.4 på sidan 224 i din förklaring.

PCA försöker hitta nya riktningar i datan där variationen är så stor som möjligt.

I figur 5.4 ser vi först ett dataset med två variabler, `X1` och `X2`.

Den första huvudkomponenten, `C1`, följer den riktning där datan varierar mest. Om datan projiceras på den riktningen kan mycket av informationen behållas trots att vi går från två dimensioner till en.

`C2` fångar variationen i den andra riktningen, men den variationen är mindre.

PCA skapar alltså nya variabler, principal components, som är kombinationer av de ursprungliga variablerna.

![Figur 5.4](./img_5/figure_5_4.png)

## Fråga 5

### Stina påstår att man alltid vill ha modeller som gör så bra prediktioner som möjligt. Kalle säger att tid också är viktigt. Vad säger du?

Kalle har en poäng.

Bäst möjliga prediktion är inte alltid det enda som spelar roll. En modell kan vara lite mer träffsäker men samtidigt ta mycket längre tid att träna eller göra prediktioner med.

I vissa system behöver prediktioner ske snabbt, exempelvis i realtid. Då kan en enklare och snabbare modell vara ett bättre val.

Det blir alltså en avvägning mellan prestanda, träningstid, prediktionstid och hur mycket resurser modellen kräver.

## Fråga 6

### Efter att vi genomfört en PCA, vad händer med tolkningen av variablerna?

Efter PCA arbetar vi inte längre direkt med de ursprungliga variablerna.

De nya variablerna, principal components, är kombinationer av flera ursprungliga features.

Det gör datan mer kompakt men också svårare att tolka. En komponent betyder exempelvis inte längre direkt "ålder", "inkomst" eller någon annan tydlig feature.

Det är alltså en nackdel med PCA om tolkbarhet är viktig.

## Fråga 8

### Förklara vad nedanstående kod gör.

In [2]:
import numpy as np
from sklearn.decomposition import PCA

# Skapar ett dataset med 1000 rader och 3 features
X = np.random.rand(1000, 3)

print(X[0:5])

# Reducerar från 3 dimensioner till 2
pca = PCA(n_components=2)
X2D = pca.fit_transform(X)

print(X2D[0:5])

# Försöker återskapa datan tillbaka till 3 dimensioner
X3D_inv = pca.inverse_transform(X2D)

# False eftersom viss information förlorades när en dimension togs bort
print(np.allclose(X3D_inv, X))

[[0.22405457 0.46596293 0.90557581]
 [0.01665019 0.15729766 0.49631512]
 [0.58033394 0.32684239 0.60771359]
 [0.23621588 0.30577353 0.46669146]
 [0.92937823 0.89766291 0.95065484]]
[[ 0.33828123 -0.16020579]
 [ 0.19846053 -0.56370788]
 [ 0.21653332  0.03004389]
 [ 0.09721152 -0.31754562]
 [ 0.15048439  0.64639124]]
False


Koden börjar med att skapa ett dataset med 1000 observationer och 3 features.

Sedan används `PCA(n_components=2)` för att reducera datan från 3 dimensioner till 2.

`fit_transform()` lär sig vilka riktningar som innehåller mest variation och transformerar sedan datan till de två nya komponenterna.

`inverse_transform()` försöker återskapa den ursprungliga datan med 3 dimensioner.

`np.allclose()` blir `False` eftersom PCA tog bort en dimension och därmed även en del information. Den ursprungliga datan går därför inte att återskapa exakt.

## Fråga 9

### Genomför en PCA på `car_price_dataset.csv` innan du modellerar det med ML. Hur påverkas resultatet?

Jag jämför samma regressionsmodell före och efter PCA.

Målet är `Price`.

In [3]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import LinearRegression
from sklearn.metrics import root_mean_squared_error, r2_score

data_path = Path("chapter_05_dimensionality_reduction/data/car_price_dataset.csv")
if not data_path.exists():
    data_path = Path("data/car_price_dataset.csv")

df = pd.read_csv(data_path, sep=";")
df.head()

,Brand,Model,Year,Engine_Size,Fuel_Type,Transmission,Mileage,Doors,Owner_Count,Price
0,Kia,Rio,2020,4.2,Diesel,Manual,289944,3,5,8501
1,Chevrolet,Malibu,2012,2.0,Hybrid,Automatic,5356,2,3,12092
2,Mercedes,GLA,2020,4.2,Diesel,Automatic,231440,4,2,11171
3,Audi,Q5,2023,2.0,Electric,Manual,160971,2,1,11780
4,Volkswagen,Golf,2003,2.6,Hybrid,Semi-Automatic,286618,3,3,2867


In [4]:
print(df.shape)
print(df.isna().sum().sum())
print(df.dtypes)

(10000, 10)
0
Brand               str
Model               str
Year              int64
Engine_Size     float64
Fuel_Type           str
Transmission        str
Mileage           int64
Doors             int64
Owner_Count       int64
Price             int64
dtype: object


In [5]:
X = df.drop(columns="Price")
y = df["Price"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

categorical_features = X.select_dtypes(include="object").columns.tolist()
numerical_features = X.select_dtypes(exclude="object").columns.tolist()

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numerical_features),
        ("cat", OneHotEncoder(
            handle_unknown="ignore",
            sparse_output=False
        ), categorical_features),
    ]
)

X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

print("Antal features före PCA:", X_train_processed.shape[1])

Antal features före PCA: 52


C:\Users\chris\AppData\Local\Temp\ipykernel_26884\1972557154.py:11: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_features = X.select_dtypes(include="object").columns.tolist()


### Modell utan PCA

In [6]:
model = LinearRegression()

model.fit(X_train_processed, y_train)
y_pred = model.predict(X_test_processed)

rmse_before = root_mean_squared_error(y_test, y_pred)
r2_before = r2_score(y_test, y_pred)

print("RMSE utan PCA:", round(rmse_before, 2))
print("R² utan PCA:", round(r2_before, 4))

RMSE utan PCA: 64.91
R² utan PCA: 0.9995


### Modell med PCA

Jag låter PCA behålla 95 % av variationen i datan.

In [7]:
pca = PCA(n_components=0.95, random_state=42)

X_train_pca = pca.fit_transform(X_train_processed)
X_test_pca = pca.transform(X_test_processed)

print("Antal features före PCA:", X_train_processed.shape[1])
print("Antal komponenter efter PCA:", X_train_pca.shape[1])
print(
    "Förklarad variation:",
    round(pca.explained_variance_ratio_.sum(), 4)
)

Antal features före PCA: 52
Antal komponenter efter PCA: 27
Förklarad variation: 0.9537


In [8]:
model_pca = LinearRegression()

model_pca.fit(X_train_pca, y_train)
y_pred_pca = model_pca.predict(X_test_pca)

rmse_after = root_mean_squared_error(y_test, y_pred_pca)
r2_after = r2_score(y_test, y_pred_pca)

print("RMSE med PCA:", round(rmse_after, 2))
print("R² med PCA:", round(r2_after, 4))

RMSE med PCA: 64.99
R² med PCA: 0.9995


In [9]:
comparison = pd.DataFrame({
    "Utan PCA": [rmse_before, r2_before],
    "Med PCA": [rmse_after, r2_after]
}, index=["RMSE", "R²"])

comparison

,Utan PCA,Med PCA
RMSE,64.914733,64.987354
R²,0.999541,0.999540


### Slutsats

PCA minskade antalet features tydligt samtidigt som resultatet nästan inte förändrades.

Det betyder att en stor del av informationen kunde representeras med färre dimensioner.

I det här fallet blev modellen alltså mer kompakt utan någon större försämring i RMSE eller R².

PCA kan därför vara användbart när man vill minska mängden data som modellen behöver arbeta med, men man förlorar samtidigt en del tolkbarhet eftersom komponenterna inte längre motsvarar de ursprungliga variablerna direkt.